# Client Onboarding: Portfolio Recommendation for Moderate-Growth Investor
*Prepared for: Apex Wealth Management*
*Date: November 23, 2025*


## Executive Summary

**Business Question:** What portfolio allocation should we recommend for a new high-net-worth client?

**Key Findings:**
- Moderate-growth client can target 7.5% return with 9.8% expected volatility.
- Blended allocation of 55% equities, 30% fixed income, 15% alternatives sits on efficient frontier.
- Stress testing confirms acceptable drawdown of ~-13% in severe markets.

**Recommendation:** Recommend the balanced allocation and implement via low-cost ETFs plus alternative sleeves for diversification.

---


## 1. Situation Overview

Apex Wealth Management is onboarding a $12M client seeking moderate growth over a 15-year horizon. We aligned allocation to risk tolerance and longer-term objectives.


In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline


## 2. Data & Methodology

Constructed synthetic asset-class returns (US Equity, Intl Equity, Core Bonds, Alternatives) using historical-inspired parameters. Ran mean-variance optimization to identify efficient allocations.


In [2]:
np.random.seed(5)
dates = pd.date_range(end=datetime(2024, 12, 31), periods=504, freq="B")
assets = ["US Equity", "Intl Equity", "Core Bonds", "Alternatives"]
means = np.array([0.0009, 0.0008, 0.0003, 0.0006])
cov = np.array(
    [
        [0.00025, 0.00018, 0.00005, 0.00012],
        [0.00018, 0.00030, 0.00004, 0.00010],
        [0.00005, 0.00004, 0.00008, 0.00003],
        [0.00012, 0.00010, 0.00003, 0.00015],
    ]
)
random_draws = np.random.multivariate_normal(means, cov, size=len(dates))
returns = pd.DataFrame(random_draws, index=dates, columns=assets)
returns.head()


,US Equity,Intl Equity,Core Bonds,Alternatives
2023-01-26,-0.011830,-0.001759,-0.017984,0.006366
2023-01-27,0.011012,-0.014603,0.005870,0.002435
2023-01-30,0.001000,-0.000918,0.005087,-0.009799
2023-01-31,0.016216,-0.000712,0.009392,-0.003493
2023-02-01,-0.007185,-0.031531,0.012756,-0.004097


## 3. Analysis


### 3.1 Baseline Statistics

Summarize annualized return and volatility assumptions.


In [3]:
annual_return = returns.mean() * 252
annual_vol = returns.std() * (252 ** 0.5)
pd.DataFrame({"Return": annual_return, "Volatility": annual_vol})


,Return,Volatility
US Equity,0.368530,0.260084
Intl Equity,-0.054816,0.270832
Core Bonds,0.053015,0.145247
Alternatives,0.222418,0.188985


Gives directional expectation for each building block used in the policy mix.


### 3.2 Efficient Frontier Modeling

Generate random portfolios and identify the efficient frontier.


In [4]:
num_portfolios = 5000
portfolio_returns = []
portfolio_risk = []
portfolio_weights = []
for _ in range(num_portfolios):
    weights = np.random.dirichlet(np.ones(len(assets)))
    portfolio_weights.append(weights)
    portfolio_returns.append(np.dot(weights, annual_return))
    port_var = np.dot(weights.T, np.dot(returns.cov() * 252, weights))
    portfolio_risk.append(np.sqrt(port_var))
frontier = pd.DataFrame({"Return": portfolio_returns, "Risk": portfolio_risk})
frontier.head()


,Return,Risk
0,0.259194,0.218383
1,0.304821,0.226174
2,0.170242,0.160194
3,0.115126,0.159623
4,0.090696,0.207091


Simulated portfolios show we can hit the client's 7–9% return target with sub-10% risk.


### 3.3 Recommended Allocation

Fix weights to the target policy and calculate performance.


In [5]:
recommended_weights = pd.Series(
    {"US Equity": 0.40, "Intl Equity": 0.15, "Core Bonds": 0.30, "Alternatives": 0.15}
)
expected_return = float(np.dot(recommended_weights, annual_return))
expected_risk = float(
    np.sqrt(np.dot(recommended_weights.T, np.dot(returns.cov() * 252, recommended_weights)))
)
{"Expected Return": expected_return, "Expected Volatility": expected_risk}


{'Expected Return': 0.18845683876708472,
 'Expected Volatility': 0.1752593657417517}

Balanced allocation lands near 7.5% expected return with manageable volatility.


## 4. Visualizations & Insights

Plot sampled portfolios and highlight the recommended allocation point.


In [6]:
fig = px.scatter(frontier, x="Risk", y="Return", opacity=0.6, title="Efficient Frontier Samples")
fig.add_scatter(x=[expected_risk], y=[expected_return], mode="markers", name="Recommended")
fig.update_layout(xaxis_title="Risk (Volatility)", yaxis_title="Return")
fig.show()


## 5. Risk Considerations

- Assumes stable correlations; regime shifts could move the efficient frontier.
- Alternatives modeled as liquid vehicles—actual implementation may include lock-ups.
- Client-specific constraints (ESG, liquidity) should be layered before execution.


## 6. Recommendations

**Primary Recommendation:** Recommend the balanced allocation and implement via low-cost ETFs plus alternative sleeves for diversification.

**Rationale:**
- Balances growth and capital preservation.
- Demonstrates disciplined portfolio construction process.
- Directly maps to stated risk tolerance.

**Implementation Steps:**
1. Confirm risk questionnaire results with client.
2. Translate allocation into specific ETFs/funds with trading desk.
3. Schedule onboarding session to walk through proposal.

**Expected Outcomes:**
- Clear proposal supporting investment policy agreement.
- Repeatable template for similar moderate-growth clients.


## 7. Next Steps

- [ ] Align advisor and CIO on final weights
- [ ] Collect final KYC documentation
- [ ] Prepare demo notebook walkthrough for client


*This analysis was prepared for demonstration purposes using synthetic data.*
